# NB-GP Normative Modeling — cfRNA Application

In [27]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gpytorch
import scanpy as sc
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse
from scipy.stats import nbinom, norm
from scipy.special import gammaln
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)


In [28]:
DATA_DIR  = Path("/project/cfRNA_NormativeModeling/OpenAccess_nfcore")
H5AD_PATH = DATA_DIR / "Merged_Processed_AnnData_with_Batch_Biases_QC_Status.h5ad"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BIAS_COLUMNS = [
    "log(Total Reads)",
    "Spliced Reads (%)",
    "gDNA Contamination (Intron/Exon)",
    "rRNA Fraction",
    "RNA Degradation (3' Bias)",
    "Platelet Score",
    "GC Bias",
    "Gene Length Bias",
    "NG80",
    "(NP80/NG80)",
]

print(f"Device   : {DEVICE}")
print(f"Data path: {H5AD_PATH}")

Device   : cuda
Data path: /project/cfRNA_NormativeModeling/OpenAccess_nfcore/Merged_Processed_AnnData_with_Batch_Biases_QC_Status.h5ad


In [29]:
adata = sc.read_h5ad(H5AD_PATH)
adata = adata[adata.obs["QC_Passed"] == True]
adata = adata[adata.obs["Phenotype_Processed"].notna()]
adata = adata[adata.obs["Phenotype_Processed"] != "Unknown"]
phenotypes = adata.obs["Phenotype_Processed"].astype(str).values
is_hc      = phenotypes == "Healthy Control"
# ── X: bias metric normalization (fit on HC only) ─────────────────
X_raw        = adata.obs[BIAS_COLUMNS].values.astype(np.float32)
scaler_X     = StandardScaler()
X_hc_scaled  = scaler_X.fit_transform(X_raw[is_hc])
X_all_scaled = scaler_X.transform(X_raw)
# ── Y: raw counts (rounded to nearest integer) ────────────────────
Y_raw = adata.X.toarray() if issparse(adata.X) else np.asarray(adata.X)
Y_raw = np.round(Y_raw).astype(np.float32)
# ── Protein-coding gene filter ────────────────────────────────────
is_pc          = (adata.var["GeneType"] == "protein_coding").values
pc_gene_names  = adata.var_names[is_pc].tolist()
pc_indices     = np.where(is_pc)[0]
# ── Disease group info ────────────────────────────────────────────
unique_diseases = sorted(set(phenotypes[~is_hc]) - {"nan", "Unknown", "None"})

print(f"Total samples  : {len(adata)}")
print(f"  HC           : {is_hc.sum()} | Disease: {(~is_hc).sum()}")
print(f"Protein-coding : {len(pc_gene_names)} genes")
print(f"X shape (HC)   : {X_hc_scaled.shape}")
print(f"Disease groups : {unique_diseases}")
print(f"Y_raw dtype    : {Y_raw.dtype}, sample values: {Y_raw[0, :5]}")

Total samples  : 3159
  HC           : 996 | Disease: 2163
Protein-coding : 20097 genes
X shape (HC)   : (996, 10)
Disease groups : ['AD', 'AML', 'CDCS', 'Colorectal Cancer', 'Diverticulitis', 'EPO_Treatment', 'Esophagus Cancer', 'GCSF_Donor', 'HIV', 'HIV + Tuberculosis', 'ICI-m ', 'ICI-treated Cancer', 'Liver Cancer', 'Liver Cirrhosis', 'Lung Cancer', 'ME/CFS', 'MGUS', 'MM', 'NAFLD', 'NASH', 'Other Cancer', 'Pancreatic Cancer', 'Pancreatic Cancer ', 'Pancreatitis', 'Pre-eclampsia', 'Stomach Cancer', 'Tuberculosis', 'Unspecified_Fibrosis']
Y_raw dtype    : float32, sample values: [ 0.  0.  0. 26. 35.]


In [30]:
# =====================================================================
# Stratified gene selection  (quantile-based bins)
# =====================================================================
# 1. Candidate filter : det_rate_hc >= 0.1  AND  mean_count_hc >= 2.0
# 2. Stratification   : 4×4 grid using quartiles of the CANDIDATE distribution
#    → each axis split at Q25/Q50/Q75 of candidates → ~equal pool per stratum
# 3. Selection        : up to k genes per stratum (random, seed-fixed)
# =====================================================================

K_PER_STRATUM = 4
GENE_SEED     = 42


def select_stratified_genes(
    Y_raw, is_hc, pc_gene_names, pc_indices,
    det_rate_min=0.1, mean_count_min=2.0,
    n_det_bins=4, n_count_bins=4,
    k=K_PER_STRATUM, seed=GENE_SEED,
):
    """
    Returns (target_names, target_indices, df_strata).
    Bin edges are derived from the quartiles of the candidate gene distribution
    so that each stratum has roughly equal numbers of candidate genes.
    """
    Y_hc   = Y_raw[is_hc][:, pc_indices]
    det_r  = (Y_hc > 0).mean(axis=0).astype(float)
    mean_c = Y_hc.mean(axis=0).astype(float)

    cand      = (det_r >= det_rate_min) & (mean_c >= mean_count_min)
    c_names   = np.array(pc_gene_names)[cand]
    c_indices = pc_indices[cand]
    c_det     = det_r[cand]
    c_mean    = mean_c[cand]

    # Quantile edges from the candidate pool
    det_q   = np.quantile(c_det,  np.linspace(0, 1, n_det_bins   + 1))
    count_q = np.quantile(c_mean, np.linspace(0, 1, n_count_bins + 1))
    # Replace the upper bound of the last bin with inf so max values are included
    det_edges   = list(det_q[:-1])   + [np.inf]
    count_edges = list(count_q[:-1]) + [np.inf]

    rng = np.random.default_rng(seed)
    sel_names, sel_indices, strata_rows = [], [], []

    for d_lo, d_hi in zip(det_edges[:-1], det_edges[1:]):
        for m_lo, m_hi in zip(count_edges[:-1], count_edges[1:]):
            in_bin = np.where(
                (c_det  >= d_lo) & (c_det  < d_hi) &
                (c_mean >= m_lo) & (c_mean < m_hi)
            )[0]
            n_avail  = len(in_bin)
            n_chosen = min(k, n_avail)
            chosen   = rng.choice(in_bin, size=n_chosen, replace=False) if n_chosen > 0 else []

            for ci in chosen:
                sel_names.append(c_names[ci])
                sel_indices.append(c_indices[ci])

            d_hi_s = f"{d_hi:.3f}" if d_hi < np.inf else "∞"
            m_hi_s = f"{m_hi:.1f}" if m_hi < np.inf else "∞"
            strata_rows.append({
                "det_bin":   f"[{d_lo:.3f}, {d_hi_s})",
                "count_bin": f"[{m_lo:.1f}, {m_hi_s})",
                "n_cand":    n_avail,
                "n_sel":     n_chosen,
            })

    df_strata = pd.DataFrame(strata_rows)
    return list(sel_names), np.array(sel_indices), df_strata


target_names, target_indices, df_strata = select_stratified_genes(
    Y_raw, is_hc, pc_gene_names, pc_indices
)

# ── Summary ──────────────────────────────────────────────────────
Y_hc_pc = Y_raw[is_hc][:, pc_indices]
n_cand  = (((Y_hc_pc > 0).mean(0) >= 0.1) & (Y_hc_pc.mean(0) >= 2.0)).sum()

print(f"Candidate genes (det≥0.1, mean≥2): {n_cand}")
print(f"Selected target genes : {len(target_names)}  "
      f"(up to {K_PER_STRATUM}/stratum × {len(df_strata)} strata)")
print()

pivot_cand = df_strata.pivot_table(
    index="count_bin", columns="det_bin", values="n_cand", aggfunc="first"
)
pivot_sel = df_strata.pivot_table(
    index="count_bin", columns="det_bin", values="n_sel", aggfunc="first"
)
print("=== n_cand (candidates per stratum) ===")
print(pivot_cand.to_string())
print("\n=== n_sel (selected per stratum) ===")
print(pivot_sel.to_string())


Candidate genes (det≥0.1, mean≥2): 15185
Selected target genes : 52  (up to 4/stratum × 16 strata)

=== n_cand (candidates per stratum) ===
det_bin        [0.102, 0.641)  [0.641, 0.915)  [0.915, 0.984)  [0.984, ∞)
count_bin                                                                
[18.0, 76.4)              259            3107             430           0
[2.0, 18.0)              3514             282               0           0
[246.4, ∞)                  4              27             445        3321
[76.4, 246.4)              16             377            2874         529

=== n_sel (selected per stratum) ===
det_bin        [0.102, 0.641)  [0.641, 0.915)  [0.915, 0.984)  [0.984, ∞)
count_bin                                                                
[18.0, 76.4)                4               4               4           0
[2.0, 18.0)                 4               4               0           0
[246.4, ∞)                  4               4               4           4
[76.4, 2

## Model Definitions

In [ ]:
# =====================================================================
# GLM: statsmodels NegativeBinomialP(p=1)
# =====================================================================
class NBGLM:
    """NB GLM wrapper. Falls back to intercept-only model on convergence failure."""

    def __init__(self):
        self._result    = None
        self._mu_fb     = None  # fallback mean
        self.theta_     = None

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        X_c = sm.add_constant(X_train, has_constant="add")
        try:
            res = sm.NegativeBinomial(y_train, X_c).fit(disp=False, maxiter=200)
            self._result = res
            
            alpha = float(res.params[-1])
            if alpha > 1e-6:
                self.theta_ = 1.0 / alpha  
            else:
                self.theta_ = 1e4  
                
        except Exception:
            self._result = None
            self._mu_fb  = float(y_train.mean()) if len(y_train) > 0 else 1.0
            self.theta_  = None  
        return self

    def predict(self, X_test: np.ndarray):
        """Returns (mu_pred: ndarray, theta: float)."""
        X_c = sm.add_constant(X_test, has_constant="add")
        if self._result is not None:
            mu = self._result.predict(X_c).astype(np.float32)
        else:
            mu = np.full(len(X_test), self._mu_fb, dtype=np.float32)
        return np.clip(mu, 1e-4, 1e6), self.theta_

In [32]:
class BayesianNBGLM(nn.Module):
    def __init__(self, n_features, theta_init=2.0, log_sigma_init=0.0):
        super().__init__()
        self.raw_theta = nn.Parameter(torch.tensor(float(np.log(np.exp(theta_init) - 1.0 + 1e-6))))
        self.log_sigma = nn.Parameter(torch.tensor(float(log_sigma_init)))

    @property
    def theta(self):
        return nn.functional.softplus(self.raw_theta) + 0.1

    @property
    def sigma(self):
        return self.log_sigma.exp()

    @staticmethod
    def _aug(X):
        """Prepend intercept column."""
        return torch.cat([torch.ones(len(X), 1, device=X.device, dtype=X.dtype), X], dim=1)

    def find_map(self, X, y, beta0=None, max_iter=100, tol=1e-7):
        """Newton MAP for beta; also returns neg_H = -∇²log p(beta|y) (positive definite)."""
        Xa = self._aug(X)
        n, p = Xa.shape
        dev = Xa.device
        theta  = float(self.theta.detach())
        sigma2 = float(self.sigma.detach().pow(2))

        beta = beta0.detach().clone() if beta0 is not None else torch.zeros(p, device=dev)

        with torch.no_grad():
            for _ in range(max_iter):
                mu       = (Xa @ beta).exp().clamp(1e-4, 1e4)
                grad_ll  = y - mu * (y + theta) / (mu + theta)
                W        = (mu * theta * (y + theta) / (mu + theta).pow(2)).clamp(1e-8)
                grad_pos = Xa.T @ grad_ll - beta / sigma2
                neg_H    = (Xa.T * W) @ Xa + torch.eye(p, device=dev) / sigma2
                try:
                    L = torch.linalg.cholesky(neg_H)
                    d = torch.cholesky_solve(grad_pos[:, None], L).squeeze()
                except Exception:
                    break
                beta_new = beta + d
                if (beta_new - beta).norm().item() < tol:
                    beta = beta_new
                    break
                beta = beta_new

            # recompute neg_H at final beta for accurate posterior covariance
            mu    = (Xa @ beta).exp().clamp(1e-4, 1e4)
            W     = (mu * theta * (y + theta) / (mu + theta).pow(2)).clamp(1e-8)
            neg_H = (Xa.T * W) @ Xa + torch.eye(p, device=dev) / sigma2

        return beta, neg_H

    def log_marginal_lik(self, X, y, beta_map, neg_H):
        """Laplace log marginal likelihood for optimising theta and sigma."""
        Xa    = self._aug(X)
        p     = Xa.shape[1]
        dev   = Xa.device
        theta  = self.theta
        sigma2 = self.sigma.pow(2)

        b  = beta_map.detach()
        mu = (Xa @ b).exp().clamp(1e-4, 1e4)

        log_lik = (
            torch.lgamma(y + theta) - torch.lgamma(theta) - torch.lgamma(y + 1)
            + theta * (theta.log() - (theta + mu).log())
            + y    * (mu.log()    - (theta + mu).log())
        ).sum()

        log_prior = -0.5 * (b.pow(2).sum() / sigma2 + p * sigma2.log())

        try:
            L       = torch.linalg.cholesky(neg_H.detach() + 1e-6 * torch.eye(p, device=dev))
            log_det = L.diagonal().log().sum()
        except Exception:
            log_det = torch.tensor(0.0, device=dev)

        return log_lik + log_prior - log_det

    def posterior_predictive(self, beta_map, neg_H, X_test):
        """Returns (f_mean, f_var) under q(beta) = N(beta_MAP, neg_H^{-1})."""
        Xa  = self._aug(X_test)
        p   = beta_map.shape[0]
        dev = Xa.device
        f_mean = Xa @ beta_map.detach()
        try:
            L     = torch.linalg.cholesky(neg_H.detach() + 1e-6 * torch.eye(p, device=dev))
            V     = torch.linalg.solve_triangular(L, Xa.T, upper=False)   # (p, N*)
            f_var = V.pow(2).sum(0).clamp(0)
        except Exception:
            f_var = torch.zeros(len(X_test), device=dev)
        return f_mean, f_var


def train_bayesian_nbglm(
    X_train, y_train,
    max_epochs=500, lr=0.01,
    patience=15, rel_tol=1e-4,
):
    dev   = X_train.device
    model = BayesianNBGLM(X_train.shape[-1]).to(dev)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    beta_prev = None

    best_loss, no_improve = float("inf"), 0
    for _ in range(max_epochs):
        model.train()
        opt.zero_grad()
        with torch.no_grad():
            beta_map, neg_H = model.find_map(X_train, y_train, beta0=beta_prev)
        beta_prev = beta_map.detach()
        try:
            loss = -model.log_marginal_lik(X_train, y_train, beta_map, neg_H)
        except Exception:
            continue
        if torch.isnan(loss):
            continue
        loss.backward()
        opt.step()

        v = loss.item()
        if best_loss == float("inf"):
            best_loss = v
            continue
        # Use absolute improvement to prevent numerical instability near zero
        if (best_loss - v) > rel_tol:
            best_loss, no_improve = v, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.eval()
    with torch.no_grad():
        beta_map, neg_H = model.find_map(X_train, y_train, beta0=beta_prev)
    return model, beta_map.detach(), neg_H.detach()


In [33]:
# =====================================================================
# Laplace Approximation NB-GP  (IFT gradient)
# =====================================================================

def _nb_grad_hess(f, y, theta):
    """NB log-likelihood gradient and diagonal Hessian magnitude W at f = log mu."""
    mu   = f.exp().clamp(1e-4, 1e4)
    grad = y - mu * (y + theta) / (mu + theta)
    W    = (mu * theta * (y + theta) / (mu + theta).pow(2)).clamp(1e-8)
    return grad, W


def _newton_map(K, y, theta_val, f0, max_iter, tol):
    """Pure-value Newton MAP iterations (no autograd tracking)."""
    n, dev = len(y), K.device
    f = f0.clone()
    for _ in range(max_iter):
        grad, W = _nb_grad_hess(f, y, theta_val)
        sqrt_W  = W.sqrt()
        B       = torch.eye(n, device=dev) + sqrt_W[:, None] * K * sqrt_W[None, :]
        for jit in [1e-4, 1e-3, 1e-2, 1e-1]:
            try:
                L = torch.linalg.cholesky(B + jit * torch.eye(n, device=dev))
                break
            except Exception:
                continue
        else:
            break
        b     = W * f + grad
        v     = torch.cholesky_solve((sqrt_W * (K @ b))[:, None], L).squeeze()
        f_new = K @ (b - sqrt_W * v)
        if (f_new - f).norm().item() < tol:
            return f_new
        f = f_new
    return f


class _FindMapIFT(torch.autograd.Function):
    """MAP finding with implicit-differentiation backward.
    Stationarity: F(K, theta, f*) = grad_f log p(y|f*,theta) - K^{-1} f* = 0
    IFT gradients:
      dL/dK     = 0.5 * (outer(K^{-1}v, alpha) + outer(alpha, K^{-1}v))  [symmetrised]
      dL/dtheta = v^T (dF/dtheta)
    where v = Sigma g, Sigma = (K^{-1} + W)^{-1}, alpha = K^{-1} f*, g = dL/df*.
    """
    @staticmethod
    def forward(ctx, K, y, theta, f0, max_iter, tol):
        with torch.no_grad():
            f = _newton_map(K.detach(), y, float(theta.detach()), f0.detach(), int(max_iter), float(tol))
        ctx.save_for_backward(K.detach(), y, theta.detach(), f)
        return f

    @staticmethod
    def backward(ctx, g):
        K, y, theta_t, f = ctx.saved_tensors
        dev = K.device
        n   = len(y)

        mu     = f.exp().clamp(1e-4, 1e4)
        W      = (mu * theta_t * (y + theta_t) / (mu + theta_t).pow(2)).clamp(1e-8)
        sqrt_W = W.sqrt()

        # v = Sigma @ g  where Sigma = (K^{-1}+W)^{-1} = K - K sqrt_W B^{-1} sqrt_W K
        B = torch.eye(n, device=dev) + sqrt_W[:, None] * K * sqrt_W[None, :]
        for jit in [1e-4, 1e-3, 1e-2]:
            try:
                L = torch.linalg.cholesky(B + jit * torch.eye(n, device=dev))
                break
            except Exception:
                continue
        else:
            return None, None, None, None, None, None

        Kg    = K @ g
        inner = torch.cholesky_solve((sqrt_W * Kg)[:, None], L).squeeze()
        v     = Kg - K @ (sqrt_W * inner)  # v = Sigma g

        # grad_K: dL/dK_{ij} = (K^{-1}v)_i * alpha_j,  symmetrised
        K_reg = K + 1e-4 * torch.eye(n, device=dev)
        for jit in [0.0, 1e-3, 1e-2]:
            try:
                K_chol = torch.linalg.cholesky(K_reg + jit * torch.eye(n, device=dev))
                break
            except Exception:
                continue
        else:
            return None, None, None, None, None, None

        alpha   = torch.cholesky_solve(f[:, None], K_chol).squeeze()   # K^{-1} f
        K_inv_v = torch.cholesky_solve(v[:, None], K_chol).squeeze()   # K^{-1} v
        grad_K  = 0.5 * (torch.outer(K_inv_v, alpha) + torch.outer(alpha, K_inv_v))

        # grad_theta: dF/dtheta = d(grad_f log p)/dtheta
        # NB: d/dtheta [y - mu(y+theta)/(mu+theta)] = -mu(mu-y)/(mu+theta)^2
        dF_dtheta  = -mu * (mu - y) / (mu + theta_t).pow(2)
        grad_theta = (v * dF_dtheta).sum()

        return grad_K, None, grad_theta, None, None, None


class LaplaceNBGP(nn.Module):
    def __init__(self, n_features, theta_init=2.0):
        super().__init__()
        self.log_output_scale = nn.Parameter(torch.zeros(1))
        self.log_length_scale = nn.Parameter(torch.zeros(n_features))
        self.raw_theta = nn.Parameter(
            torch.tensor(float(np.log(np.exp(theta_init) - 1.0 + 1e-6)))
        )

    @property
    def theta(self):
        return nn.functional.softplus(self.raw_theta) + 0.1

    def kernel(self, X1, X2=None):
        if X2 is None:
            X2 = X1
        ls   = self.log_length_scale.exp()
        os   = self.log_output_scale.exp()
        diff = (X1.unsqueeze(1) - X2.unsqueeze(0)) / ls
        return os * (-0.5 * diff.pow(2).sum(-1)).exp()

    def _safe_cholesky(self, A):
        dev = A.device
        for jitter in [1e-4, 1e-3, 1e-2, 1e-1]:
            try:
                return torch.linalg.cholesky(A + jitter * torch.eye(len(A), device=dev))
            except Exception:
                continue
        raise RuntimeError("Cholesky failed after jitter escalation")

    def find_map(self, K, y, f0=None, max_iter=50, tol=1e-5):
        """MAP via Newton iteration; backward uses implicit differentiation."""
        n, dev = len(y), K.device
        if f0 is None:
            f0 = torch.zeros(n, device=dev)
        return _FindMapIFT.apply(K, y, self.theta, f0.detach(), max_iter, tol)

    def log_marginal_lik(self, K, f_map, y):
        """Laplace log marginal likelihood. f_map is kept live — gradient flows via IFT."""
        theta = self.theta
        f     = f_map                        # no .detach(); grad flows through IFT
        dev   = K.device

        mu     = f.exp().clamp(1e-4, 1e4)
        W      = (mu * theta * (y + theta) / (mu + theta).pow(2)).clamp(1e-8)
        sqrt_W = W.sqrt()

        B = torch.eye(len(y), device=dev) + sqrt_W[:, None] * K * sqrt_W[None, :]
        try:
            L = self._safe_cholesky(B)
        except RuntimeError:
            return K.sum() * 0.0 - 1e8

        log_lik = (
            torch.lgamma(y + theta) - torch.lgamma(theta) - torch.lgamma(y + 1)
            + theta * (theta.log() - (theta + mu).log())
            + y    * (mu.log()    - (theta + mu).log())
        ).sum()

        K_chol    = self._safe_cholesky(K)
        alpha     = torch.cholesky_solve(f[:, None], K_chol).squeeze()
        log_prior = -0.5 * (f * alpha).sum()
        log_det   = -L.diagonal().log().sum()
        return log_lik + log_prior + log_det

    def posterior_predictive(self, K_train, f_map, y, K_cross, k_test_diag):
        f   = f_map.detach()
        dev = K_train.device
        theta = self.theta.detach()

        _, W   = _nb_grad_hess(f, y, theta)
        sqrt_W = W.sqrt()
        B      = torch.eye(len(y), device=dev) + sqrt_W[:, None] * K_train * sqrt_W[None, :]
        try:
            L = self._safe_cholesky(B)
        except RuntimeError:
            return torch.zeros(k_test_diag.shape, device=dev), k_test_diag

        # f_mean = K(X*,X) K^{-1} f_map  — direct linear solve, no stationarity assumption
        K_chol = self._safe_cholesky(K_train)
        alpha  = torch.cholesky_solve(f[:, None], K_chol).squeeze()
        f_mean = K_cross.T @ alpha

        v     = torch.linalg.solve_triangular(L, sqrt_W[:, None] * K_cross, upper=False)
        f_var = (k_test_diag - v.pow(2).sum(0)).clamp(0)
        return f_mean, f_var


def train_laplace_nbgp(
    X_train, y_train,
    max_epochs=500, lr=0.01,
    patience=15, rel_tol=1e-4,
):
    """Train LaplaceNBGP with IFT gradient. Returns (model, f_map) in eval mode.
    Stops early when relative log-marginal-likelihood improvement < rel_tol
    for `patience` consecutive epochs."""
    dev    = X_train.device
    model  = LaplaceNBGP(X_train.shape[-1]).to(dev)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    f_prev = torch.zeros(len(y_train), device=dev)

    best_loss, no_improve = float("inf"), 0
    for _ in range(max_epochs):
        model.train()
        opt.zero_grad()
        K     = model.kernel(X_train)                       # live tensor for gradient
        f_map = model.find_map(K, y_train, f0=f_prev)      # IFT-backed MAP
        try:
            loss = -model.log_marginal_lik(K, f_map, y_train)
        except Exception:
            f_prev = f_map.detach()
            continue
        if torch.isnan(loss):
            f_prev = f_map.detach()
            continue
        loss.backward()
        opt.step()
        f_prev = f_map.detach()

        v = loss.item()
        if best_loss == float("inf"):
            best_loss = v
            continue
        # Use absolute improvement to prevent numerical instability near zero
        if (best_loss - v) > rel_tol:
            best_loss, no_improve = v, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.eval()
    with torch.no_grad():
        K     = model.kernel(X_train)
        f_map = _newton_map(K, y_train, float(model.theta.detach()), f_prev, 50, 1e-5)
    return model, f_map.detach()


## Z-score Functions

두 가지 방식을 모두 제공하며, `ZSCORE_METHOD` 변수로 선택한다.

| 방식 | 공식 | 특성 |
|------|------|------|
| `"pearson"` | $(y - \mu) / \sqrt{\mu + \mu^2/\theta}$ | 재현 가능, 점근적 N(0,1) |
| `"quantile"` | $\Phi^{-1}(U[F(y-1), F(y)])$ | 엄밀히 N(0,1), stochastic |

In [34]:
def pearson_zscore(y: np.ndarray, mu: np.ndarray, theta: float) -> np.ndarray:
    """Pearson residual under NB(mu, theta). Asymptotically N(0,1). Reproducible."""
    var = mu + mu ** 2 / max(theta, 1e-6)
    return (y - mu) / np.sqrt(var + 1e-8)


def quantile_zscore(
    y: np.ndarray, mu: np.ndarray, theta: float, seed: int | None = None
) -> np.ndarray:
    """Randomized quantile residual (Dunn & Smyth 1996).
    Exactly N(0,1) under the correct NB model.
    Stochastic — pass seed for reproducibility within a run.

    scipy nbinom(n, p) parameterization: p = theta / (theta + mu)
    """
    theta = max(theta, 1e-4)
    p     = np.clip(theta / (theta + mu + 1e-8), 1e-8, 1 - 1e-8)
    y_int = y.astype(int)

    # CDF at y-1: for y=0, CDF(-1) = 0 by convention
    a = np.where(y_int > 0, nbinom.cdf(y_int - 1, n=theta, p=p), 0.0)
    b = nbinom.cdf(y_int, n=theta, p=p)

    rng = np.random.default_rng(seed)
    u   = rng.uniform(
        np.clip(a, 1e-8, 1 - 1e-8),
        np.clip(b, 1e-8, 1 - 1e-8),
    )
    return norm.ppf(u)


def compute_zscore(
    y: np.ndarray,
    mu: np.ndarray,
    theta: float,
    method: str = "pearson",
    seed: int | None = None,
) -> np.ndarray:
    """method: 'pearson' | 'quantile'"""
    y, mu = np.asarray(y, dtype=np.float32), np.asarray(mu, dtype=np.float32)
    if method == "pearson":
        return pearson_zscore(y, mu, theta)
    if method == "quantile":
        return quantile_zscore(y, mu, theta, seed=seed)
    raise ValueError(f"Unknown method: {method!r}. Choose 'pearson' or 'quantile'.")


# ── Per-model scoring helpers ─────────────────────────────────────

def score_glm(
    glm: NBGLM, X_all: np.ndarray, y_all: np.ndarray,
    method: str = "pearson", seed: int | None = None,
):
    """Returns (z, mu_pred, theta)."""
    mu, theta = glm.predict(X_all)
    return compute_zscore(y_all, mu, theta, method=method, seed=seed), mu, theta


def score_bayes_glm(
    bglm: BayesianNBGLM, beta_map: torch.Tensor, neg_H: torch.Tensor,
    X_all_t: torch.Tensor, y_all: np.ndarray,
    method: str = "pearson", n_samples: int = 300, seed: int | None = None,
):
    """MC scoring by sampling beta ~ q(beta) = N(beta_MAP, neg_H^{-1}).

    Samples the full coefficient vector (dim p=11), then projects to f* = X*_aug @ beta.
    Avoids the N*×N* covariance matrix; memory cost is only p×S.
    """
    if seed is not None:
        torch.manual_seed(seed)
    with torch.no_grad():
        Xa  = bglm._aug(X_all_t)
        p   = beta_map.shape[0]
        dev = beta_map.device

        L      = torch.linalg.cholesky(neg_H + 1e-6 * torch.eye(p, device=dev))
        eps    = torch.randn(p, n_samples, device=dev)
        # Sigma_q = L^{-1} L^{-T}  →  sample: beta = beta_MAP + L^{-T} eps
        d_beta = torch.linalg.solve_triangular(L.T, eps, upper=True)   # (p, S)
        betas  = beta_map[:, None] + d_beta                             # (p, S)
        f_samp = Xa @ betas                                             # (N_all, S)

    mu    = f_samp.exp().mean(1).cpu().numpy()
    theta = bglm.theta.item()
    return compute_zscore(y_all, mu, theta, method=method, seed=seed), mu, theta


def score_laplace(
    lap_model: LaplaceNBGP, f_map: torch.Tensor,
    X_tr_t: torch.Tensor, y_tr_t: torch.Tensor,
    X_all_t: torch.Tensor, y_all: np.ndarray,
    method: str = "pearson", n_samples: int = 300, seed: int | None = None,
):
    """Returns (z, mu_pred, theta). mu_pred estimated via Laplace posterior MC."""
    with torch.no_grad():
        K_tr    = lap_model.kernel(X_tr_t)
        K_cross = lap_model.kernel(X_tr_t, X_all_t)
        k_diag  = lap_model.kernel(X_all_t).diagonal()
        f_mean, f_var = lap_model.posterior_predictive(K_tr, f_map, y_tr_t, K_cross, k_diag)
    f_samples = torch.distributions.Normal(
        f_mean, f_var.sqrt().clamp(1e-6)
    ).rsample(torch.Size([n_samples]))
    mu    = f_samples.exp().mean(0).cpu().numpy()
    theta = lap_model.theta.item()
    return compute_zscore(y_all, mu, theta, method=method, seed=seed), mu, theta


## Gene-by-Gene Pipeline

In [35]:
X_tr_t  = torch.tensor(X_hc_scaled,  dtype=torch.float32).to(DEVICE)
X_all_t = torch.tensor(X_all_scaled, dtype=torch.float32).to(DEVICE)


In [36]:
# ── Run Configuration ────────────────────────────────────────────
ZSCORE_METHOD = "pearson"   # "pearson" | "quantile"
RANDOM_SEED   = 42

# Upper-bound epochs — training stops earlier via patience-based convergence
BGLM_MAX_EPOCHS = 300
LAP_MAX_EPOCHS  = 300
MC_SAMPLES      = 500

N_all = len(adata)

print(f"Z-score method : {ZSCORE_METHOD}")
print(f"Device         : {DEVICE}")
print(f"Target genes   : {len(target_names)}  (stratified selection)")
print(f"HC samples     : {is_hc.sum()} | All samples: {N_all}")
print(f"Early stopping : patience=15 (BGLM/Lap), rel_tol=1e-4")

# ── Storage ──────────────────────────────────────────────────────
z_glm_all  = np.full((N_all, len(target_names)), np.nan, dtype=np.float32)
z_bglm_all = np.full((N_all, len(target_names)), np.nan, dtype=np.float32)
z_lap_all  = np.full((N_all, len(target_names)), np.nan, dtype=np.float32)
gene_meta  = []


def _sync():
    """Flush GPU queue before reading the wall clock."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# ── Gene loop ────────────────────────────────────────────────────
for i, (g_name, g_idx) in enumerate(zip(target_names, target_indices)):
    y_hc  = Y_raw[is_hc,  g_idx]
    y_all = Y_raw[:, g_idx]
    y_tr_t = torch.tensor(y_hc, dtype=torch.float32).to(DEVICE)

    det_rate_hc   = float((y_hc > 0).mean())
    hc_mean_count = float(y_hc.mean())

    meta = {
        "gene": g_name,
        "det_rate_hc": det_rate_hc,
        "hc_mean_count": hc_mean_count,
    }

    # 1. GLM (point estimate, linear)
    _sync(); t0 = time.perf_counter()
    glm = NBGLM().fit(X_hc_scaled, y_hc)
    z_glm, _, th_glm = score_glm(
        glm, X_all_scaled, y_all, method=ZSCORE_METHOD, seed=RANDOM_SEED
    )
    _sync(); meta["time_glm_s"] = time.perf_counter() - t0
    z_glm_all[:, i] = z_glm
    meta["theta_glm"] = th_glm

    # 2. Bayesian NB-GLM (posterior, linear)
    _sync(); t0 = time.perf_counter()
    bglm, beta_map, neg_H = train_bayesian_nbglm(
        X_tr_t, y_tr_t, max_epochs=BGLM_MAX_EPOCHS
    )
    z_bglm, _, th_bglm = score_bayes_glm(
        bglm, beta_map, neg_H, X_all_t, y_all,
        method=ZSCORE_METHOD, n_samples=MC_SAMPLES, seed=RANDOM_SEED,
    )
    _sync(); meta["time_bglm_s"] = time.perf_counter() - t0
    z_bglm_all[:, i] = z_bglm
    meta["theta_bglm"] = th_bglm
    del bglm, beta_map, neg_H

    # 3. Laplace NB-GP (posterior, nonlinear, IFT gradient)
    _sync(); t0 = time.perf_counter()
    lap_model, f_map = train_laplace_nbgp(
        X_tr_t, y_tr_t, max_epochs=LAP_MAX_EPOCHS
    )
    z_lap, _, th_lap = score_laplace(
        lap_model, f_map, X_tr_t, y_tr_t, X_all_t, y_all,
        method=ZSCORE_METHOD, n_samples=MC_SAMPLES, seed=RANDOM_SEED,
    )
    _sync(); meta["time_lap_s"] = time.perf_counter() - t0
    z_lap_all[:, i] = z_lap
    meta["theta_lap"] = th_lap
    del lap_model, f_map

    gene_meta.append(meta)

    print(
        f"[{i+1:3d}/{len(target_names)}] {g_name:<22s} "
        f"det={det_rate_hc:.2f} mean={hc_mean_count:6.1f}  "
        f"θ: glm={th_glm:.2f} bglm={th_bglm:.2f} lap={th_lap:.2f}  |  "
        f"t: {meta['time_glm_s']:.1f}+{meta['time_bglm_s']:.1f}+"
        f"{meta['time_lap_s']:.1f}s"
    )


Z-score method : pearson
Device         : cuda
Target genes   : 52  (stratified selection)
HC samples     : 996 | All samples: 3159
Early stopping : patience=15 (BGLM/Lap), rel_tol=1e-4
[  1/52] ENSG00000158292.8      det=0.53 mean=  11.2  θ: glm=0.10 bglm=0.52 lap=1.25  |  t: 0.1+8.1+21.8s
[  2/52] ENSG00000100276.11     det=0.25 mean=   3.6  θ: glm=0.10 bglm=0.42 lap=0.61  |  t: 0.0+7.2+21.6s
[  3/52] ENSG00000179761.13     det=0.64 mean=  12.6  θ: glm=0.10 bglm=0.55 lap=1.19  |  t: 0.0+7.9+22.0s
[  4/52] ENSG00000196826.7      det=0.21 mean=   2.9  θ: glm=0.10 bglm=0.42 lap=0.65  |  t: 0.0+6.5+22.4s
[  5/52] ENSG00000250021.7      det=0.62 mean=  52.2  θ: glm=0.10 bglm=0.55 lap=0.77  |  t: 0.0+15.8+22.1s
[  6/52] ENSG00000127325.20     det=0.63 mean=  18.4  θ: glm=0.10 bglm=0.56 lap=0.71  |  t: 0.0+10.2+22.4s


KeyboardInterrupt: 

In [ ]:
df_meta = pd.DataFrame(gene_meta)

time_cols = ["time_glm_s", "time_bglm_s", "time_lap_s"]
print(f"\nDone. Z-score matrices: {z_glm_all.shape}")
print("\nTraining time summary (seconds):")
print(df_meta[time_cols].describe().loc[["mean", "std", "min", "max"]].round(2).to_string())

## QC Metrics & Gene Filtering

HC 샘플의 Z-score 분포를 기준으로 유효 유전자를 선별한다.

In [ ]:
def compute_hc_qc(
    z_matrix: np.ndarray, is_hc: np.ndarray, gene_names: list
) -> pd.DataFrame:
    """Z-score QC on HC samples per gene."""
    z_hc   = z_matrix[is_hc]
    z_mean = np.nanmean(z_hc, axis=0)
    z_std  = np.nanstd(z_hc,  axis=0)
    return pd.DataFrame({"gene": gene_names, "Z_Mean": z_mean, "Z_Std": z_std})


def filter_valid_genes(
    df_meta: pd.DataFrame,
    df_qc: pd.DataFrame,
    z_mean_th: float = 0.3,       # |Z_Mean| ≤ 0.3
    z_std_min: float = 0.5,       # Z_Std ∈ [0.5, 1.5]
    z_std_max: float = 1.5,
) -> pd.DataFrame:
    df = df_meta.merge(df_qc, on="gene")

    m_zmean = df["Z_Mean"].abs()  <= z_mean_th
    m_zstd  = (df["Z_Std"] >= z_std_min) & (df["Z_Std"] <= z_std_max)

    df["valid"]       =  m_zmean & m_zstd
    df["fail_reason"] = ""
    df.loc[~m_zmean, "fail_reason"] += "Biased_Mean "
    df.loc[df["Z_Std"] < z_std_min, "fail_reason"] += "Low_Std(Overfit) "
    df.loc[df["Z_Std"] > z_std_max, "fail_reason"] += "High_Std(Underfit) "
    df["fail_reason"] = df["fail_reason"].str.strip()
    return df


# ── Compute QC ────────────────────────────────────────────────────
qc_glm  = compute_hc_qc(z_glm_all,  is_hc, target_names)
qc_bglm = compute_hc_qc(z_bglm_all, is_hc, target_names)
qc_lap  = compute_hc_qc(z_lap_all,  is_hc, target_names)

df_filter_glm  = filter_valid_genes(df_meta, qc_glm)
df_filter_bglm = filter_valid_genes(df_meta, qc_bglm)
df_filter_lap  = filter_valid_genes(df_meta, qc_lap)

for m_name, df_f in [
    ("GLM",       df_filter_glm),
    ("Bayes-GLM", df_filter_bglm),
    ("Laplace-GP",df_filter_lap),
]:
    n_valid = df_f["valid"].sum()
    print(f"{m_name:11s} — valid: {n_valid}/{len(df_f)} ({n_valid/len(df_f)*100:.1f}%)")
    reason_counts = (
        df_f.loc[~df_f["valid"] & df_f["fail_reason"].ne(""), "fail_reason"]
        .value_counts().head(5)
    )
    print(reason_counts.to_string())
    print()


## Results & Visualization

In [ ]:
# ── 1. HC Z-score calibration: Z_Mean & Z_Std distributions (2 × 3) ─
method_configs = [
    ("GLM",        qc_glm,  "#F44336"),
    ("Bayes-GLM",  qc_bglm, "#FF9800"),
    ("Laplace-GP", qc_lap,  "#2196F3"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plt.suptitle(f"HC Z-score Calibration — 3 Models ({ZSCORE_METHOD})", fontsize=14, fontweight="bold")

for col, (m_name, qc_df, color) in enumerate(method_configs):
    ax = axes[0, col]
    ax.hist(qc_df["Z_Mean"].dropna(), bins=64, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(0, color="k", lw=1.5, ls="--")
    ax.set_title(f"{m_name}\nZ_Mean", fontsize=9)
    ax.set_xlabel("Mean Z-score (HC)"); ax.set_ylabel("# genes")

    ax = axes[1, col]
    ax.hist(qc_df["Z_Std"].dropna(), bins=40, color=color, alpha=0.75, edgecolor="white")
    ax.set_title(f"{m_name}\nZ_Std", fontsize=9)
    ax.set_xlabel("Std Z-score (HC)"); ax.set_ylabel("# genes")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()
pairs_layout = [
    ("GLM",       qc_glm,  "Bayes-GLM",  qc_bglm, "Linear: point → posterior"),
    ("GLM",       qc_glm,  "Laplace-GP", qc_lap,  "GLM vs Laplace-GP"),
    ("Bayes-GLM", qc_bglm, "Laplace-GP", qc_lap,  "Bayes-GLM vs Laplace-GP"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plt.suptitle("HC Z_Mean: pairwise model comparison", fontsize=13, fontweight="bold")

for ax, (n1, df1, n2, df2, note) in zip(axes.flat, pairs_layout):
    m = df1.merge(df2, on="gene", suffixes=(f"_{n1.replace('-','')}", f"_{n2.replace('-','')}"))
    c1 = f"Z_Mean_{n1.replace('-','')}"
    c2 = f"Z_Mean_{n2.replace('-','')}"
    ax.scatter(m[c1], m[c2], s=5, alpha=0.5)
    lim = max(m[[c1, c2]].abs().max().max(), 0.5) * 1.1
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.4)
    ax.set_xlabel(f"Z_Mean ({n1})"); ax.set_ylabel(f"Z_Mean ({n2})")
    ax.set_title(note, fontsize=8)

plt.tight_layout()
plt.show()

# ── 3. Valid-gene summary bar chart ───────────────────────────────
n_genes = len(target_names)
summary = {
    "GLM":        df_filter_glm["valid"].sum(),
    "Bayes-GLM":  df_filter_bglm["valid"].sum(),
    "Laplace-GP": df_filter_lap["valid"].sum(),
}
colors_bar = [c for _, _, c in method_configs]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(summary.keys(), summary.values(), color=colors_bar, alpha=0.85, edgecolor="white")
ax.set_ylim(0, n_genes * 1.15)
ax.axhline(n_genes, color="k", lw=0.8, ls="--", label=f"Total ({n_genes})")
for bar, v in zip(bars, summary.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
            f"{v}\n({v/n_genes*100:.0f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("# valid genes")
ax.set_title(f"Valid genes after QC filtering (n={n_genes})", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ── Save Z-score matrices & QC tables ────────────────────────────
sample_ids = adata.obs_names.tolist()
OUT_DIR    = Path("/project/cfRNA_NormativeModeling/results_nb_gp")
OUT_DIR.mkdir(exist_ok=True)

tag = f"{ZSCORE_METHOD}_n{len(target_names)}"

for z_mat, label in [
    (z_glm_all,  "glm"),
    (z_bglm_all, "bayes_glm"),
    (z_lap_all,  "laplace"),
]:
    pd.DataFrame(z_mat, index=sample_ids, columns=target_names).to_csv(
        OUT_DIR / f"zscores_{label}_{tag}.csv"
    )

for df_f, label in [
    (df_filter_glm,  "glm"),
    (df_filter_bglm, "bayes_glm"),
    (df_filter_lap,  "laplace"),
]:
    df_f.to_csv(OUT_DIR / f"qc_{label}_{tag}.csv", index=False)

df_meta.to_csv(OUT_DIR / f"timing_{tag}.csv", index=False)

print(f"Saved to: {OUT_DIR}")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")
